# Telecom X - Análise de Evasão de Clientes

## Objetivo
Este notebook executa o processo de ETL e a análise exploratória dos dados de clientes da Telecom X para identificar padrões associados à evasão.

## Perguntas de negócio
- Qual é a proporção de clientes que evadem?
- Quais perfis contratuais e comportamentais concentram maior risco de evasão?
- Como as variáveis categóricas e numéricas se comportam entre clientes que ficaram e clientes que saíram?

## Etapas do trabalho
1. Extração dos dados em JSON e normalização para DataFrame.
2. Limpeza, padronização, tradução de colunas e criação da variável Contas_Diarias.
3. Análise descritiva e visual da evasão.
4. Relatório final com conclusões e recomendações.

## Observação sobre a fonte
O notebook está preparado para ler uma API quando a URL estiver disponível. Neste workspace, o arquivo local é usado como fallback para garantir execução imediata.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)

API_URL = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json"
LOCAL_JSON_PATH = Path("TelecomX_Data.json")
DATA_DICTIONARY_PATH = Path("TelecomX_dicionario.md")


def carregar_dados(api_url: str, local_path: Path) -> tuple[list[dict], str]:
    try:
        response = requests.get(api_url, timeout=30)
        response.raise_for_status()
        return response.json(), f"API: {api_url}"
    except requests.RequestException as exc:
        print(f"Falha ao carregar os dados pela API: {exc}")
        print("Arquivo local utilizado como fallback.")
        with local_path.open(encoding="utf-8") as arquivo:
            return json.load(arquivo), f"Arquivo local: {local_path}"


dados_brutos, fonte_dados = carregar_dados(API_URL, LOCAL_JSON_PATH)
df_bruto = pd.json_normalize(dados_brutos, sep="_")

colunas_esperadas = [
    "customerID",
    "Churn",
    "customer_gender",
    "customer_SeniorCitizen",
    "customer_Partner",
    "customer_Dependents",
    "customer_tenure",
    "phone_PhoneService",
    "phone_MultipleLines",
    "internet_InternetService",
    "internet_OnlineSecurity",
    "internet_OnlineBackup",
    "internet_DeviceProtection",
    "internet_TechSupport",
    "internet_StreamingTV",
    "internet_StreamingMovies",
    "account_Contract",
    "account_PaperlessBilling",
    "account_PaymentMethod",
    "account_Charges_Monthly",
    "account_Charges_Total",
]

display(Markdown("## Extração e normalização"))
display(Markdown(f"**Fonte utilizada:** {fonte_dados}"))
display(Markdown(f"- Registros brutos: **{len(df_bruto)}**\n- Colunas após normalização: **{df_bruto.shape[1]}**"))
display(df_bruto[colunas_esperadas].head())
print("Tipos de dados iniciais:")
display(df_bruto.dtypes.to_frame("tipo_dado"))

# Transformação e tratamento

Nesta etapa, os dados são preparados para análise:

- colunas aninhadas são renomeadas para nomes mais claros;
- variáveis categóricas são traduzidas para português;
- registros inconsistentes são tratados;
- a coluna Contas_Diarias é criada a partir da cobrança mensal;
- as variáveis mais relevantes para a evasão são destacadas com apoio do dicionário de dados.

In [ ]:
dicionario_texto = DATA_DICTIONARY_PATH.read_text(encoding="utf-8")
display(Markdown("## Dicionário de dados"))
display(Markdown(dicionario_texto))

mapa_colunas = {
    "customerID": "IdCliente",
    "Churn": "Evasao",
    "customer_gender": "Genero",
    "customer_SeniorCitizen": "Idoso",
    "customer_Partner": "Parceiro",
    "customer_Dependents": "Dependentes",
    "customer_tenure": "Meses_Contrato",
    "phone_PhoneService": "Servico_Telefonico",
    "phone_MultipleLines": "Multiplas_Linhas",
    "internet_InternetService": "Servico_Internet",
    "internet_OnlineSecurity": "Seguranca_Online",
    "internet_OnlineBackup": "Backup_Online",
    "internet_DeviceProtection": "Protecao_Dispositivo",
    "internet_TechSupport": "Suporte_Tecnico",
    "internet_StreamingTV": "Streaming_TV",
    "internet_StreamingMovies": "Streaming_Filmes",
    "account_Contract": "Tipo_Contrato",
    "account_PaperlessBilling": "Fatura_Digital",
    "account_PaymentMethod": "Metodo_Pagamento",
    "account_Charges_Monthly": "Gasto_Mensal",
    "account_Charges_Total": "Gasto_Total",
}

mapa_valores = {
    "Yes": "Sim",
    "No": "Não",
    "Female": "Feminino",
    "Male": "Masculino",
    "Month-to-month": "Mensal",
    "One year": "Um ano",
    "Two year": "Dois anos",
    "Electronic check": "Cheque eletrônico",
    "Mailed check": "Cheque enviado",
    "Bank transfer (automatic)": "Transferência bancária automática",
    "Credit card (automatic)": "Cartão de crédito automático",
    "Fiber optic": "Fibra óptica",
    "No internet service": "Sem internet",
    "No phone service": "Sem telefone",
}

df = df_bruto.rename(columns=mapa_colunas).copy()
df["Gasto_Total"] = pd.to_numeric(df["Gasto_Total"], errors="coerce")
df["Evasao"] = df["Evasao"].replace("", pd.NA)

df = df.replace(mapa_valores)

gasto_total_nulo_antes = int(df["Gasto_Total"].isna().sum())
df["Gasto_Total"] = df["Gasto_Total"].fillna(df["Gasto_Mensal"] * df["Meses_Contrato"])

df_limpo = df.dropna(subset=["Evasao"]).copy()
df_limpo["Contas_Diarias"] = df_limpo["Gasto_Mensal"] / 30
df_limpo["Evasao_Binaria"] = df_limpo["Evasao"].map({"Sim": 1, "Não": 0})
df_limpo["Idoso"] = df_limpo["Idoso"].map({1: "Sim", 0: "Não"})

variaveis_relevantes = [
    "Evasao",
    "Genero",
    "Idoso",
    "Parceiro",
    "Dependentes",
    "Meses_Contrato",
    "Tipo_Contrato",
    "Metodo_Pagamento",
    "Servico_Internet",
    "Gasto_Mensal",
    "Gasto_Total",
    "Contas_Diarias",
]

resumo_tratamento = pd.DataFrame(
    {
        "Métrica": [
            "Registros originais",
            "Registros com evasão vazia removidos",
            "Valores nulos em Gasto_Total corrigidos",
            "Registros prontos para análise",
        ],
        "Valor": [
            len(df_bruto),
            int(df["Evasao"].isna().sum()),
            gasto_total_nulo_antes,
            len(df_limpo),
        ],
    }
)

display(Markdown("## Tratamento e padronização"))
display(resumo_tratamento)
print("Tipos de dados após o tratamento:")
display(df_limpo[variaveis_relevantes].dtypes.to_frame("tipo_dado"))
display(Markdown("## Variáveis mais relevantes para a análise de evasão"))
display(pd.DataFrame({"Variável": variaveis_relevantes, "Justificativa": [
    "Variável alvo da análise",
    "Permite verificar diferença de evasão por perfil demográfico",
    "Ajuda a medir risco em clientes de maior faixa etária",
    "Indica estabilidade familiar do cliente",
    "Complementa o contexto familiar",
    "Tempo de relacionamento com a empresa",
    "Relação direta com fidelização e compromisso",
    "Pode refletir atrito no processo de cobrança",
    "Mostra o tipo de serviço contratado",
    "Mede o valor recorrente pago pelo cliente",
    "Representa o histórico financeiro acumulado",
    "Aproxima o gasto do cliente em granularidade diária",
]}))
display(df_limpo[variaveis_relevantes].head())

# Análise exploratória de dados

Esta seção reúne estatísticas descritivas e visualizações para entender a distribuição da evasão e seus principais fatores associados.

As análises são divididas em:
- distribuição geral da variável alvo;
- comportamento da evasão por variáveis categóricas;
- comparação entre variáveis numéricas de clientes que ficaram e clientes que saíram.

In [ ]:
display(Markdown("## Análise descritiva"))
metricas_numericas = df_limpo[["Meses_Contrato", "Gasto_Mensal", "Gasto_Total", "Contas_Diarias"]].describe().T
metricas_numericas["mediana"] = df_limpo[["Meses_Contrato", "Gasto_Mensal", "Gasto_Total", "Contas_Diarias"]].median()
display(metricas_numericas)

percentual_evasao = (df_limpo["Evasao_Binaria"].mean() * 100)
display(Markdown(f"## Distribuição da evasão\nA taxa de evasão observada na base tratada é de **{percentual_evasao:.2f}%**."))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

evasao_counts = df_limpo["Evasao"].value_counts().rename(index={"Não": "Permaneceu", "Sim": "Evadido"})
axes[0, 0].pie(evasao_counts.values, labels=evasao_counts.index, autopct="%1.1f%%", startangle=90)
axes[0, 0].set_title("Proporção de evasão")

contrato_churn = (
    df_limpo.groupby("Tipo_Contrato", observed=False)["Evasao_Binaria"]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .reset_index()
)
sns.barplot(data=contrato_churn, x="Tipo_Contrato", y="Evasao_Binaria", hue="Tipo_Contrato", legend=False, ax=axes[0, 1])
axes[0, 1].set_title("Taxa de evasão por tipo de contrato")
axes[0, 1].set_xlabel("Tipo de contrato")
axes[0, 1].set_ylabel("Taxa de evasão (%)")

pagamento_churn = (
    df_limpo.groupby("Metodo_Pagamento", observed=False)["Evasao_Binaria"]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .reset_index()
)
sns.barplot(data=pagamento_churn, y="Metodo_Pagamento", x="Evasao_Binaria", hue="Metodo_Pagamento", legend=False, ax=axes[1, 0])
axes[1, 0].set_title("Taxa de evasão por método de pagamento")
axes[1, 0].set_xlabel("Taxa de evasão (%)")
axes[1, 0].set_ylabel("Método de pagamento")

sns.boxplot(data=df_limpo, x="Evasao", y="Meses_Contrato", hue="Evasao", legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Meses de contrato por evasão")
axes[1, 1].set_xlabel("Evasão")
axes[1, 1].set_ylabel("Meses de contrato")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.countplot(data=df_limpo, x="Genero", hue="Evasao", ax=axes[0, 0])
axes[0, 0].set_title("Evasão por gênero")
axes[0, 0].set_xlabel("Gênero")
axes[0, 0].set_ylabel("Quantidade")

sns.countplot(data=df_limpo, x="Idoso", hue="Evasao", ax=axes[0, 1])
axes[0, 1].set_title("Evasão por faixa sênior")
axes[0, 1].set_xlabel("Cliente idoso")
axes[0, 1].set_ylabel("Quantidade")

sns.countplot(data=df_limpo, x="Servico_Internet", hue="Evasao", ax=axes[1, 0])
axes[1, 0].set_title("Evasão por tipo de internet")
axes[1, 0].set_xlabel("Serviço de internet")
axes[1, 0].set_ylabel("Quantidade")

sns.boxplot(data=df_limpo, x="Evasao", y="Gasto_Mensal", hue="Evasao", legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Gasto mensal por evasão")
axes[1, 1].set_xlabel("Evasão")
axes[1, 1].set_ylabel("Gasto mensal")

plt.tight_layout()
plt.show()

insights_categoricos = pd.DataFrame(
    {
        "Indicador": [
            "Taxa de evasão geral",
            "Maior risco por contrato",
            "Maior risco por pagamento",
            "Maior risco por internet",
            "Taxa de evasão em clientes idosos",
        ],
        "Valor": [
            f"{percentual_evasao:.2f}%",
            f"{contrato_churn.iloc[0, 0]} ({contrato_churn.iloc[0, 1]:.2f}%)",
            f"{pagamento_churn.iloc[0, 0]} ({pagamento_churn.iloc[0, 1]:.2f}%)",
            f"{df_limpo.groupby('Servico_Internet')['Evasao_Binaria'].mean().mul(100).sort_values(ascending=False).index[0]} ({df_limpo.groupby('Servico_Internet')['Evasao_Binaria'].mean().mul(100).sort_values(ascending=False).iloc[0]:.2f}%)",
            f"{df_limpo[df_limpo['Idoso'] == 'Sim']['Evasao_Binaria'].mean() * 100:.2f}%",
        ],
    }
)

display(Markdown("## Principais indicadores da EDA"))
display(insights_categoricos)

comparativo_numerico = (
    df_limpo.groupby("Evasao")[["Meses_Contrato", "Gasto_Mensal", "Gasto_Total", "Contas_Diarias"]]
    .mean()
    .round(2)
)
display(Markdown("## Comparação média entre clientes que ficaram e clientes que saíram"))
display(comparativo_numerico)

# Relatório final

## Introdução
A Telecom X enfrenta um cenário de evasão de clientes e precisa entender os fatores que mais contribuem para cancelamentos. Este notebook consolidou a extração, o tratamento e a análise exploratória dos dados para apoiar decisões de retenção.

## Limpeza e tratamento de dados
Os dados foram carregados de uma fonte JSON remota, normalizados para estrutura tabular e revisados com apoio do dicionário de dados. Em seguida, foram tratadas inconsistências de evasão em branco, convertidos valores numéricos, padronizados textos, traduzidas colunas e criada a variável Contas_Diarias.

## Análise exploratória
A análise combinou estatística descritiva com gráficos de proporção, contagem e distribuição, permitindo comparar o comportamento de clientes evadidos e não evadidos em variáveis categóricas e numéricas.

## Objetivo do relatório
Consolidar os principais achados da análise e apontar recomendações práticas para redução da evasão.

In [ ]:
taxa_contrato = df_limpo.groupby("Tipo_Contrato")["Evasao_Binaria"].mean().mul(100).round(2).sort_values(ascending=False)
taxa_pagamento = df_limpo.groupby("Metodo_Pagamento")["Evasao_Binaria"].mean().mul(100).round(2).sort_values(ascending=False)
taxa_internet = df_limpo.groupby("Servico_Internet")["Evasao_Binaria"].mean().mul(100).round(2).sort_values(ascending=False)
media_tempo = df_limpo.groupby("Evasao")["Meses_Contrato"].mean().round(2)
media_gasto_mensal = df_limpo.groupby("Evasao")["Gasto_Mensal"].mean().round(2)
media_gasto_total = df_limpo.groupby("Evasao")["Gasto_Total"].mean().round(2)

relatorio = f"""
## Conclusões e insights
- A base tratada ficou com **{len(df_limpo)} clientes válidos** e taxa geral de evasão de **{df_limpo['Evasao_Binaria'].mean() * 100:.2f}%**.
- O maior risco de evasão aparece em contratos **{taxa_contrato.index[0]}**, com **{taxa_contrato.iloc[0]:.2f}%**.
- O método de pagamento com maior evasão foi **{taxa_pagamento.index[0]}**, com **{taxa_pagamento.iloc[0]:.2f}%**.
- Entre os serviços de internet, **{taxa_internet.index[0]}** apresentou a maior taxa de evasão, com **{taxa_internet.iloc[0]:.2f}%**.
- Clientes que evadiram apresentam, em média, **{media_tempo['Sim']:.2f} meses de contrato**, enquanto os que permaneceram têm **{media_tempo['Não']:.2f} meses**.
- O gasto mensal médio é maior entre clientes evadidos (**{media_gasto_mensal['Sim']:.2f}**) do que entre clientes retidos (**{media_gasto_mensal['Não']:.2f}**), sugerindo possível percepção de baixo custo-benefício.
- Mesmo com gasto mensal maior, o gasto total acumulado dos clientes evadidos é menor (**{media_gasto_total['Sim']:.2f}**) porque eles permanecem menos tempo na base.

## Recomendações
1. Priorizar ações de retenção para clientes com contrato mensal, especialmente nos primeiros meses de relacionamento.
2. Revisar a jornada de cobrança e suporte de clientes que utilizam cheque eletrônico, pois esse grupo concentra o maior risco de evasão.
3. Avaliar ofertas e qualidade percebida para clientes de fibra óptica, que apresentam evasão superior aos demais perfis de internet.
4. Criar campanhas específicas para clientes idosos e para clientes com alto gasto mensal e baixo tempo de permanência.
5. Usar as variáveis tratadas deste notebook como base para a próxima etapa de modelagem preditiva de churn.
"""

display(Markdown(relatorio))